## Process Messages from the Queue published by Event Grid

### Installing Libraries and Utilities

In [ ]:
%pip install azure-servicebus==7.14.3 openai==2.38.0 python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# loading service bus configurations
service_bus_namespace = os.getenv("SERVICE_BUS_NAMESPACE")
service_bus_connection_string = os.getenv("SERVICE_BUS_CONNECTION_STRING")
service_bus_queue_name = os.getenv("SERVICE_BUS_QUEUE_NAME")

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model = os.getenv("CHAT_COMPLETIONS_MODEL")

### Creating the Service Bus Client

In [ ]:
from azure.servicebus import ServiceBusClient

sb_client = ServiceBusClient.from_connection_string(
    conn_str = service_bus_connection_string
)

### Helper function to process user queries

In [ ]:
from openai import AzureOpenAI

def generate_image_description(image_url, llm):
    azure_openai_client = AzureOpenAI(
        azure_endpoint = azure_openai_endpoint,
        api_version = "2024-06-01",
        api_key = azure_openai_api_key
    )

    response = azure_openai_client.chat.completions.create(
        model = llm,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI assistant"
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Describe this image"
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_url
                        }
                    }
                ]
            }
        ],
        temperature = 0.7
    )

    return response.choices[0].message.content

### Process messages is peek-lock mode reliably

In [ ]:
from azure.servicebus import ServiceBusReceiveMode
import json

# create a queue receiver object to reliably process messages
receiver = sb_client.get_queue_receiver(
    queue_name = service_bus_queue_name,
    receive_mode = ServiceBusReceiveMode.PEEK_LOCK,
    max_wait_time = 60
)

for msg in receiver:
    try:
        payload = json.loads(str(msg))

        # extracting the image url from the payload
        image_url = payload["data"]["url"]

        print("processing image url: {}".format(image_url))
        print("\n")
        assistant_response = generate_image_description(image_url, chat_completions_model)
        print("Assistant Reponse: {}".format(assistant_response))
        print("============================================")
        print("\n")

        # Mark the message as complete after successful execution
        receiver.complete_message(msg)
    except Exception as e:
        receiver.dead_letter_message(
            msg,
            reason = "Invalid Image URL",
            error_description = "Invalid Image URL"
        )
